# Notebook 2: Fine-tuning de Modelos YOLO para Detección de EPP

**Trabajo Práctico – Visión por Computadora II**  

---

## Objetivos
1. Fine-tunear **YOLOv8n** (nano) sobre Construction-PPE
2. Fine-tunear **YOLOv8m** (medium) para comparación
3. Fine-tunear **YOLOv11n** (nano) para evaluar nueva arquitectura
4. Guardar pesos y métricas para el Notebook de Comparación

> **💡 Nota CPU:** En modo CPU se recomienda:
> - Reducir `epochs` a 10–20 para exploración inicial
> - Reducir `imgsz` a 416
> - Usar `workers=0` para evitar problemas de multiprocessing en Windows

## 1. Configuración

In [ ]:
import json
import torch
from pathlib import Path
from ultralytics import YOLO
import yaml

# Cargar configuración del dataset (creada en Notebook 1)
with open("../data/dataset_config.json") as f:
    ds_config = json.load(f)

DATASET_YAML = ds_config["dataset_yaml"]
CLASS_NAMES  = ds_config["class_names"]
NUM_CLASSES  = ds_config["num_classes"]

# Parámetros de entrenamiento
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE  = 640 if DEVICE == 'cuda' else 416    # reducido en CPU
BATCH     = 16  if DEVICE == 'cuda' else 4       # menor batch en CPU
EPOCHS    = 50  if DEVICE == 'cuda' else 15      # menos epochs en CPU
WORKERS   = 4   if DEVICE == 'cuda' else 0       # 0 workers en CPU/Windows

RUNS_DIR  = Path("../runs")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

print(f"Dispositivo:    {DEVICE}")
print(f"Imagen size:    {IMG_SIZE}")
print(f"Batch size:     {BATCH}")
print(f"Epochs:         {EPOCHS}")
print(f"Dataset YAML:   {DATASET_YAML}")
print(f"Clases ({NUM_CLASSES}): {CLASS_NAMES}")

## 2. Entrenamiento YOLOv8n (Nano)

El modelo más liviano. Ideal para dispositivos edge y comparación de velocidad.

In [ ]:
print("=" * 60)
print("  Entrenando YOLOv8n (nano)")
print("=" * 60)

model_v8n = YOLO('yolov8n.pt')   # descarga automática de pesos preentrenados en COCO

results_v8n = model_v8n.train(
    data     = DATASET_YAML,
    epochs   = EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = BATCH,
    device   = DEVICE,
    workers  = WORKERS,
    project  = str(RUNS_DIR),
    name     = "yolov8n_ppe",
    exist_ok = True,
    patience = 10,           # early stopping
    save     = True,
    plots    = True,         # genera curvas de entrenamiento
    verbose  = True,
)

# Copiar mejor modelo
best_v8n = RUNS_DIR / "yolov8n_ppe" / "weights" / "best.pt"
import shutil
shutil.copy(best_v8n, MODELS_DIR / "yolov8n_best.pt")
print(f"\n✅ YOLOv8n guardado en: {MODELS_DIR / 'yolov8n_best.pt'}")

## 3. Entrenamiento YOLOv8m (Medium)

Mayor capacidad pero más lento. En CPU puede demorar significativamente más.

In [ ]:
print("=" * 60)
print("  Entrenando YOLOv8m (medium)")
print("=" * 60)

if DEVICE == 'cpu':
    print("⚠️  En CPU este entrenamiento puede tardar varias horas.")
    print("   Considerá usar Google Colab con GPU para este modelo.")

model_v8m = YOLO('yolov8m.pt')

results_v8m = model_v8m.train(
    data     = DATASET_YAML,
    epochs   = EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = max(2, BATCH // 2),   # batch más chico para model mediano en CPU
    device   = DEVICE,
    workers  = WORKERS,
    project  = str(RUNS_DIR),
    name     = "yolov8m_ppe",
    exist_ok = True,
    patience = 10,
    save     = True,
    plots    = True,
    verbose  = True,
)

best_v8m = RUNS_DIR / "yolov8m_ppe" / "weights" / "best.pt"
shutil.copy(best_v8m, MODELS_DIR / "yolov8m_best.pt")
print(f"\n✅ YOLOv8m guardado en: {MODELS_DIR / 'yolov8m_best.pt'}")

## 4. Entrenamiento YOLOv11n (Nano v11)

Nueva arquitectura con mejoras en cuello de botella y cabezal de detección.

In [ ]:
print("=" * 60)
print("  Entrenando YOLOv11n (YOLO11 nano)")
print("=" * 60)

model_v11n = YOLO('yolo11n.pt')   # Ultralytics YOLO11

results_v11n = model_v11n.train(
    data     = DATASET_YAML,
    epochs   = EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = BATCH,
    device   = DEVICE,
    workers  = WORKERS,
    project  = str(RUNS_DIR),
    name     = "yolov11n_ppe",
    exist_ok = True,
    patience = 10,
    save     = True,
    plots    = True,
    verbose  = True,
)

best_v11n = RUNS_DIR / "yolov11n_ppe" / "weights" / "best.pt"
shutil.copy(best_v11n, MODELS_DIR / "yolov11n_best.pt")
print(f"\n✅ YOLOv11n guardado en: {MODELS_DIR / 'yolov11n_best.pt'}")

## 5. Evaluación en Validation Set

In [ ]:
# Evaluar los tres modelos en validation set
import time

model_paths = {
    'YOLOv8n': MODELS_DIR / 'yolov8n_best.pt',
    'YOLOv8m': MODELS_DIR / 'yolov8m_best.pt',
    'YOLOv11n': MODELS_DIR / 'yolov11n_best.pt',
}

eval_results = {}

for model_name, model_path in model_paths.items():
    if not model_path.exists():
        print(f"⚠️  {model_name}: no encontrado, saltando...")
        continue
    
    print(f"\nEvaluando {model_name}...")
    model = YOLO(str(model_path))
    
    start = time.time()
    metrics = model.val(
        data    = DATASET_YAML,
        imgsz   = IMG_SIZE,
        device  = DEVICE,
        workers = WORKERS,
        verbose = False,
    )
    val_time = time.time() - start
    
    eval_results[model_name] = {
        'map50':    round(float(metrics.box.map50), 4),
        'map50_95': round(float(metrics.box.map), 4),
        'precision': round(float(metrics.box.mp), 4),
        'recall':   round(float(metrics.box.mr), 4),
        'val_time_s': round(val_time, 1),
    }
    print(f"  mAP50:      {eval_results[model_name]['map50']:.4f}")
    print(f"  mAP50-95:   {eval_results[model_name]['map50_95']:.4f}")
    print(f"  Precision:  {eval_results[model_name]['precision']:.4f}")
    print(f"  Recall:     {eval_results[model_name]['recall']:.4f}")

# Guardar resultados para Notebook 3
with open("../data/eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

print("\n✅ Resultados guardados en data/eval_results.json")

## 6. Benchmark de Latencia de Inferencia

In [ ]:
import numpy as np
import cv2

# Imagen de prueba sintética (simula latencia real)
dummy_img = np.random.randint(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
N_RUNS = 20

latency_results = {}

for model_name, model_path in model_paths.items():
    if not model_path.exists():
        continue
    
    model = YOLO(str(model_path))
    
    # Warm-up
    for _ in range(3):
        _ = model.predict(dummy_img, verbose=False, device=DEVICE)
    
    # Medición
    times = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        _ = model.predict(dummy_img, verbose=False, device=DEVICE)
        times.append((time.perf_counter() - t0) * 1000)  # ms
    
    latency_results[model_name] = {
        'mean_ms':   round(np.mean(times), 1),
        'std_ms':    round(np.std(times), 1),
        'p50_ms':    round(np.percentile(times, 50), 1),
        'p95_ms':    round(np.percentile(times, 95), 1),
        'fps':       round(1000 / np.mean(times), 1),
    }
    print(f"{model_name}: {latency_results[model_name]['mean_ms']:.1f} ms ± {latency_results[model_name]['std_ms']:.1f} | {latency_results[model_name]['fps']:.1f} FPS")

# Guardar latencias
with open("../data/latency_results.json", "w") as f:
    json.dump(latency_results, f, indent=2)
    
print("\n✅ Latencias guardadas en data/latency_results.json")

---
## ✅ Checklist Notebook 2
- [ ] YOLOv8n entrenado y guardado
- [ ] YOLOv8m entrenado y guardado
- [ ] YOLOv11n entrenado y guardado
- [ ] Métricas de validación guardadas en eval_results.json
- [ ] Benchmark de latencia guardado en latency_results.json

**Próximo paso:** Notebook 3 – Comparación de Arquitecturas